
## Term Project - Milestone 4

### Connecting to an API/Pulling in the Data and Cleaning/Formatting

Perform at least 5 data transformation and/or cleansing steps to your API data. The below examples are not required - they are just potential transformations you could do. If your data doesn't work for these scenarios, complete different transformations. You can do the same transformation multiple times if needed to clean your data. The goal is a clean dataset at the end of the milestone. As a reminder - you cannot export your API data to CSV to work with it, you must do all the work directly against the API/JSON source.

Examples:

- Replace Headers
- Format data into a more readable format
- Identify outliers and bad data
- Find duplicates
- Fix casing or inconsistent values
- Conduct Fuzzy Matching

Make sure you clearly number and label each transformation step (Step #1, Step #2, etc.) in your code and describe what it is doing in 1-2 sentences.

First I will bring in my milestone 2 data, the original csv of data so I will have the titles of the movies that I will be querying for in the API.

In [13]:
# First I want to get my milestone 2 data so I have the name of the titles I want to fetch
# libraries
import pandas as pd
import json
import ast
import re

# load the TMDb 5000 Movies data into a dataframe
movies = pd.read_csv('tmdb_5000_movies.csv')

# display the first few rows of data
movies.head(2)

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2007-05-19,961000000,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500


Then I need to normalize the movie titles, removing any random characters, spaces, and convert to lower case to try to help in finding the name of the movie in the api. 

In [14]:
# function to normalize the movie titles
def normalize_title(title):
    # convert to lowercase
    title = title.lower()
    # remove special characters (keep only letters and numbers)
    title = re.sub(r'[^a-z0-9\s]', '', title)
    # remove extra internal spaces
    title = re.sub(r'\s+', ' ', title).strip()
    return title

# apply normalization
# create a Simplified_Title column for the stripped version and use the pretty Movie_Title column for displays
movies['Simplified_Title'] = movies['title'].apply(normalize_title)

Now I am ready to fetch data from the API. The OMDB API has a limit of 1000 hits a day for my free key so I will need to spend a few days collecting the data. While I collect the data I want to save it to a file and then be able to re-start my pull again the next day and add to my file. 
So I have created a progress file for saving data as I pull it. I will add it to an existing dataframe and then for all days after day 1 pull I will read from my existing dataframe, see where I left off, and then create a new dataframe of movies I fetch for the day and then concat the previous data and the new data and then save to a csv for the next day.

In [15]:
import pandas as pd
import os
import time
import json
import urllib.request
import urllib.parse

# setup to reach out to OMDB API
with open('APIkeys.json') as f:
    keys = json.load(f)
omdbapi = keys['OMDBapi']

serviceurl = 'http://www.omdbapi.com/?'
apikey = '&apikey=' + omdbapi

# storage file for fetching titles 1000/day
progress_file = 'omdb_data.csv'

# first day I will need to create the file, subsequent days I want to use the same file
if os.path.exists(progress_file):
    df_existing = pd.read_csv(progress_file)
    # Get a list of titles we already have
    already_fetched = df_existing['Movie_Title'].apply(normalize_title).unique().tolist()
    print(f"Resuming: Already have {len(already_fetched)} movies in {progress_file}.")
else:
    df_existing = pd.DataFrame()
    already_fetched = []
    print("Starting fresh: No previous data found.")

# identify movies that still need to be fetched, my milestone 2 data is just over 4800
all_titles = movies['Simplified_Title'].unique().tolist()
remaining_titles = [t for t in all_titles if t not in already_fetched]

# limit daily to 975 to stay under my 1000 limit
today_batch = remaining_titles[:975]
print(f"Prepared to fetch {len(today_batch)} new movies today.")

new_records = []

# loop to fetch movies
for movie in today_batch:
    try:
        url = serviceurl + urllib.parse.urlencode({'t': movie}) + apikey
        with urllib.request.urlopen(url) as response:
            data = json.loads(response.read().decode())
            if data.get('Response') == 'True':
                new_records.append(data)
        time.sleep(0.1) # respectful delay
    except Exception as e:
        print(f"Error on {movie}: {e}")

# save and add to dataframe
if new_records:
    df_new = pd.DataFrame(new_records)
    
    # Standardize headers immediately so the append is clean
    df_new = df_new.rename(columns={'Title': 'Movie_Title', 'Year': 'Release_Year'})
    
    # Combine old data with new data
    df_updated = pd.concat([df_existing, df_new], ignore_index=True)
    
    # Save back to CSV for tomorrow
    df_updated.to_csv(progress_file, index=False)
    print(f"Success! Total records in {progress_file}: {len(df_updated)}")
else:
    print("No new records were added.")


Resuming: Already have 4480 movies in omdb_data.csv.
Prepared to fetch 405 new movies today.
Success! Total records in omdb_data.csv: 5326


Now that I have a bunch of movies, I am ready to take the csv I created of them and load that into a new dataframe to do my transformations.

In [18]:
# load the OMDB Movies data into a dataframe
omdb = pd.read_csv('omdb_data.csv')

# display the first few rows of data
omdb.head(2)

,Movie_Title,Release_Year,Rated,Released,Runtime,Genre,Director,Writer,Actors,Plot,...,imdbRating,imdbVotes,imdbID,Type,DVD,BoxOffice,Production,Website,Response,totalSeasons
0,Avatar,2009,PG-13,18 Dec 2009,162 min,"Action, Adventure, Fantasy",James Cameron,James Cameron,"Sam Worthington, Zoe Saldaña, Sigourney Weaver",A paraplegic Marine dispatched to the moon Pan...,...,7.9,"1,486,308",tt0499549,movie,NaN,"$785,221,649",NaN,NaN,True,NaN
1,Spectre,2015,PG-13,06 Nov 2015,148 min,"Action, Adventure, Thriller",Sam Mendes,"John Logan, Neal Purvis, Robert Wade","Daniel Craig, Christoph Waltz, Léa Seydoux",A cryptic message from James Bond's past sends...,...,6.8,"493,010",tt2379713,movie,NaN,"$200,074,609",NaN,NaN,True,NaN


I will begin by pairing down the columns- I do not need 26 columns and that will just take time to do any tasks with this dataframe so I will pick only the ones that I believe will be relevant for this project and keep those.
After selecting my columns to keep I will rename the columns to be more in line with my other datasets for this project.

In [19]:
# Step 1 - Pairing down columns and standardizing headers

# List of columns to keep
columns_to_keep = [
    'Movie_Title', 'Release_Year', 'Released', 'Genre', 
    'Awards', 'Ratings', 'Metascore', 'imdbRating', 
    'imdbVotes', 'BoxOffice'
]


# Create the new cleaned dataframe by selecting only the specific columns
omdb_cleaned = omdb[columns_to_keep].copy()

# Rename specific headers for consistency with my other datasets
omdb_cleaned = omdb_cleaned.rename(columns={
    'imdbRating': 'IMDb_Rating',
    'imdbVotes': 'IMDb_Votes',
    'BoxOffice': 'API_Box_Office',
    'Metascore': 'Metacritic_Score'
})

print("*** Step #1 Complete: ***\nDataset filtered to 10 key columns and headers standardized.")
omdb_cleaned.head(2)

*** Step #1 Complete: ***
Dataset filtered to 10 key columns and headers standardized.


,Movie_Title,Release_Year,Released,Genre,Awards,Ratings,Metacritic_Score,IMDb_Rating,IMDb_Votes,API_Box_Office
0,Avatar,2009,18 Dec 2009,"Action, Adventure, Fantasy",Won 3 Oscars. 91 wins & 131 nominations total,"[{'Source': 'Internet Movie Database', 'Value'...",83.0,7.9,"1,486,308","$785,221,649"
1,Spectre,2015,06 Nov 2015,"Action, Adventure, Thriller",Won 1 Oscar. 8 wins & 37 nominations total,"[{'Source': 'Internet Movie Database', 'Value'...",60.0,6.8,"493,010","$200,074,609"


Next is ensuring that all numeric data is actually numeric. This means I want to make sure the release year is a datetime, the scores and ratings should be numbers, and I want to take the IMDB votes and box office numers and make them a numer and remove the commas so I have just numbers.

In [20]:
# Step 2 - Format and Clean Numeric data

# convert Release_Year to a datetime object
# This allows for easier time-based filtering later
omdb_cleaned['Release_Year'] = pd.to_datetime(omdb_cleaned['Release_Year'], errors='coerce')

# convert Metacritic_Score and IMDb_Rating to numeric (floats/ints)
omdb_cleaned['Metacritic_Score'] = pd.to_numeric(omdb_cleaned['Metacritic_Score'], errors='coerce')
omdb_cleaned['IMDb_Rating'] = pd.to_numeric(omdb_cleaned['IMDb_Rating'], errors='coerce')

# clean and convert IMDb_Votes
# We must remove the commas (e.g., "1,234") before converting to a number
omdb_cleaned['IMDb_Votes'] = pd.to_numeric(
    omdb_cleaned['IMDb_Votes'].str.replace(',', '', regex=False), 
    errors='coerce'
)

# clean and convert API_Box_Office
# strip the '$' and ',' characters (e.g., "$1,000,000" becomes "1000000")
omdb_cleaned['API_Box_Office'] = pd.to_numeric(
    omdb_cleaned['API_Box_Office'].str.replace(r'[\$,]', '', regex=True), 
    errors='coerce'
)

print("*** Step #2 Complete:***  \nNumeric and date columns formatted.")
# Display info to verify the Dtype changes
omdb_cleaned.info(2)

*** Step #2 Complete:***  
Numeric and date columns formatted.
<class 'pandas.DataFrame'>
RangeIndex: 5326 entries, 0 to 5325
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Movie_Title       5326 non-null   str           
 1   Release_Year      5225 non-null   datetime64[us]
 2   Released          5230 non-null   str           
 3   Genre             5234 non-null   str           
 4   Awards            4052 non-null   str           
 5   Ratings           5326 non-null   str           
 6   Metacritic_Score  3947 non-null   float64       
 7   IMDb_Rating       4819 non-null   float64       
 8   IMDb_Votes        4914 non-null   float64       
 9   API_Box_Office    3980 non-null   float64       
dtypes: datetime64[us](1), float64(4), str(5)
memory usage: 416.2 KB


Now I want to handle any nulls values. I have 5144 movies in this dataset and the only column that shows all 5144 as non-null is the Movie_title. This means I have other columns that do have null values so I want to fill any null with n/a to make them easier to find and get counts. 

In [21]:
# Step 3 - Handle inconsistent values

# Fill all missing values (NaN/Null) with the string "N/A"
omdb_cleaned = omdb_cleaned.fillna("N/A")

# Calculate the number of "N/A" entries for every column
na_counts = (omdb_cleaned == "N/A").sum()

print("*** Step #3 Complete: *** \nNulls standardized to 'N/A' strings.")
print("-" * 30)
print("Count of 'N/A' values per column:")
print(na_counts)

*** Step #3 Complete: *** 
Nulls standardized to 'N/A' strings.
------------------------------
Count of 'N/A' values per column:
Movie_Title            0
Release_Year         101
Released              96
Genre                 92
Awards              1274
Ratings                0
Metacritic_Score    1379
IMDb_Rating          507
IMDb_Votes           412
API_Box_Office      1346
dtype: int64


The Ratings column has some good information in it, but it is a nested object like so:

```	Ratings
[
    {'Source': 'Internet Movie Database', 'Value': '8.4/10'},
    {'Source': 'Rotten Tomatoes', 'Value': '87%'},
    {'Source': 'Metacritic', 'Value': '78/100'}
]
```
I would like to pull the Rotten Tomatoes score out, since that is a typical value to judge success of a movie. I want to save this to a new column so I can reference it easily and I also want to know how many of the movies do not have a rotten tomatoes value listed.


In [22]:
# Step 4 - Extracting rotten tomatoes scores from the nested "Ratings" strings

import ast

def extract_rt(rating_val):
    # If it was already marked as N/A in Step 3, skip it
    if rating_val == "N/A":
        return "N/A"
    
    try:
        # Convert the string representation of a list into an actual Python list
        # We use str(rating_val) to handle any unexpected types
        ratings_list = ast.literal_eval(str(rating_val))
        
        for r in ratings_list:
            if r['Source'] == 'Rotten Tomatoes':
                # Strip the '%' and return as a number
                return int(r['Value'].replace('%', ''))
    except (ValueError, SyntaxError):
        return "N/A"
    
    return "N/A"

# Apply the function to create the new column
omdb_cleaned['RT_Score'] = omdb_cleaned['Ratings'].apply(extract_rt)

# Count how many RT scores we actually found vs how many are N/A
rt_count = (omdb_cleaned['RT_Score'] != "N/A").sum()
rt_na = (omdb_cleaned['RT_Score'] == "N/A").sum()

print(f"*** Step #4 Complete: *** \nRotten Tomatoes scores extracted.")
print(f"Successfully found {rt_count} RT scores. Missing: {rt_na}")

*** Step #4 Complete: *** 
Rotten Tomatoes scores extracted.
Successfully found 4174 RT scores. Missing: 1152


I would also like to do something with the Awards data column. This would be a nice column to make visualizations with and easily see if a movie had gotten awards, and if they did was it an Oscar, only nominated, or hadn't received any awards. I will create categories for the awards data and show how many movies fall into which categories.

In [23]:
# Step 5 - Standardizing and Simplifying Awards Data

def categorize_awards(award_str):
    award_str = str(award_str).lower()
    
    if "n/a" in award_str:
        return "No Awards"
    elif "oscar" in award_str and "won" in award_str:
        return "Oscar Winner"
    elif "won" in award_str or "win" in award_str:
        return "Award Winner"
    elif "nomination" in award_str:
        return "Nominated Only"
    else:
        return "No Awards"

# apply the categorization logic
omdb_cleaned['Award_Status'] = omdb_cleaned['Awards'].apply(categorize_awards)

# show the distribution of the new categories
print("*** Step #5 Complete: *** \nAwards standardized into categories.")
print("-" * 30)
print(omdb_cleaned['Award_Status'].value_counts())

*** Step #5 Complete: *** 
Awards standardized into categories.
------------------------------
Award_Status
Award Winner      2783
No Awards         1274
Nominated Only     906
Oscar Winner       363
Name: count, dtype: int64


Now I need my simplified title that I will use for joining all the movies together later. This is my consistent title across all of the datasources.

In [24]:
# function to normalize the movie titles
def normalize_title(title):
    # convert to lowercase
    title = title.lower()
    # remove special characters (keep only letters and numbers)
    title = re.sub(r'[^a-z0-9\s]', '', title)
    # remove extra internal spaces
    title = re.sub(r'\s+', ' ', title).strip()
    return title

# apply normalization
# create a Simplified_Title column for the stripped version and use the pretty Movie_Title column for displays
omdb_cleaned['Simplified_Title'] = omdb_cleaned['Movie_Title'].apply(normalize_title)

print("*** Step #6 Complete: \nComparison of Original vs. Simplified Title:***")
print(omdb_cleaned[['Movie_Title', 'Simplified_Title']].head(5))

*** Step #6 Complete: 
Comparison of Original vs. Simplified Title:***
                                         Movie_Title  \
0                                             Avatar   
1                                            Spectre   
2                              The Dark Knight Rises   
3                                        John Carter   
4  Fighting, Flying and Driving: The Stunts of Sp...   

                                    Simplified_Title  
0                                             avatar  
1                                            spectre  
2                              the dark knight rises  
3                                        john carter  
4  fighting flying and driving the stunts of spid...  


Creating my final review of the data

In [25]:
# Final review of the cleaned dataset
print("*** Final Cleaned Dataset Overview: ***")
display(omdb_cleaned[['Simplified_Title', 'RT_Score', 'Award_Status', 'API_Box_Office']].head(10))

# Check the data types one last time
print("\nFinal Data Types:")
print(omdb_cleaned.dtypes)

*** Final Cleaned Dataset Overview: ***


,Simplified_Title,RT_Score,Award_Status,API_Box_Office
0,avatar,81,Oscar Winner,785221649.0
1,spectre,63,Oscar Winner,200074609.0
2,the dark knight rises,87,Award Winner,448149584.0
3,john carter,52,Award Winner,73078100.0
4,fighting flying and driving the stunts of spid...,N/A,No Awards,N/A
5,tangled,89,Award Winner,200821936.0
6,avengers age of ultron,75,Award Winner,459005868.0
7,batman v superman dawn of justice,28,Award Winner,330360194.0
8,superman returns,72,Award Winner,200081192.0
9,quantum of solace,63,Award Winner,168368427.0



Final Data Types:
Movie_Title            str
Release_Year        object
Released               str
Genre                  str
Awards                 str
Ratings                str
Metacritic_Score    object
IMDb_Rating         object
IMDb_Votes          object
API_Box_Office      object
RT_Score            object
Award_Status           str
Simplified_Title       str
dtype: object


Saving the data to a csv for easier loading and use later.

In [26]:
# save to csv
omdb_cleaned.to_csv('omdb_cleaned.csv')


**Ethical Implications of data wrangling for OMDb API**

The data for this milestone was acquired ethically through the OMDb API, utilizing an authorized API key and implementing a batch-processing logic with a 0.1 second time delay between requests to ensure compliance with server rate limit and to prevent a Denial of Service (DoS) impact. This data is from OBDb API which is recognized as highly credible for personal, educational, and hobby projects. The data is sourced from major platforms and authoritative entertainment sites like IMDb, Rotten Tomatoes, and Metacritic. I cross-referenced the movies with the TMDb master list from my Milestone 2 submission to ensure I only pulled the needed movies. For data wrangingling I standardized headers, formatted numeric/date types, standardized null values to `N/A`, extracted nested Rotten Tomatoes scores, categorized the Award values and created a simplified movie title. The broad assumption I made was for categorizing the awards ino only four buckets to represent the success of a film. This could create a risk of information loss, since the specific number of wins and the distinction between minor and major awards are gone. To mitigate this risk, the original Awards column is still available for review. It is also worth noting that there was missing financial data for the box office numbers, and with converting nulls to `N/A` this could create a selection bias if I did any modeling, since these could be from older or independent films that may not have the same level of detail available and could therefore be excluded from financial correlations. To mitigate this risk I have maintained a copy of the original data and documented exact counts for missing values for transparency. Legally, I am only using this data for educational purposes and I am not trying to replicate the API's commercial service.